# WP2b: Sentinel-1 Pre-processing (SAR / Hornsund Track)

**Owner: Julian**

Speckle filtering and GLCM texture feature computation following Williams & Swirad (2025).
Produces a **10-band composite per image** (2 backscatter + 8 GLCM) that feeds the SVM in `03b`.

| Band | Description |
|---|---|
| HH, HV | Raw backscatter (dB) |
| HH_var, HH_contrast, HH_ent, HH_asm | GLCM texture on HH |
| HV_var, HV_contrast, HV_ent, HV_asm | GLCM texture on HV |

In [ ]:
import ee
import geemap
import sys
sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project
from src.preprocessing_s1 import filter_dual_pol, preprocess_s1, COMPOSITE_BANDS

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

## 2b.1 Load S1 dual-pol collection

Filters to IW mode with both HH and HV bands. A server-side check (`filter_dual_pol`)
drops the rare single-pol IW images that slip through GEE's metadata filter.

In [ ]:
START, END = '2019-01-01', '2024-12-31'

s1_raw = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HH'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HV'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select(['HH', 'HV'])
)
s1_raw = filter_dual_pol(s1_raw)
print('S1 dual-pol images:', s1_raw.size().getInfo())

## 2b.2 Speckle filter + GLCM features

1. Focal-mean speckle filter (50 m radius)
2. GLCM computed on integer-scaled backscatter (9 × 9 window)
3. Four texture features selected per band: variance, contrast, entropy, ASM

In [ ]:
s1 = s1_raw.map(preprocess_s1)
print('Bands per image:', s1.first().bandNames().getInfo())

## 2b.3 Inspect a sample composite

In [ ]:
sample = s1.first().clip(aoi)
print('Sample date:', sample.date().format('YYYY-MM-dd').getInfo())

Map = geemap.Map()
Map.centerObject(aoi, zoom=9)
Map.addLayer(sample.select('HH'), {'min': -25, 'max': 0, 'palette': ['black', 'white']}, 'HH backscatter')
Map.addLayer(sample.select('HV'), {'min': -30, 'max': -5, 'palette': ['black', 'white']}, 'HV backscatter')
Map.addLayer(sample.select('HH_ent'), {'min': 0, 'max': 3}, 'HH Entropy')
Map.addLayer(sample.select('HV_var'), {'min': 0, 'max': 5000}, 'HV Variance')
Map

## Notes

- GLCM window size (`GLCM_SIZE = 4`) corresponds to a 9 × 9 pixel kernel at 50 m → ~450 m spatial context. Tune if needed.
- The 10-band composite is the direct input to the SVM in `03b_classification_s1.ipynb`.
- **Training labels** are digitised in `03b` using S2 true-colour as a visual reference.